# Human Expert Framing Labels Analysis

This notebook processes the collected human expert framing labels. The main goal is to keep only sentences where annotators agree on the framing label, so the resulting human labels can be used to correct or calibrate LLM-based sentence framing for government AI policy strategies.

In [27]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != 'AI-Policy' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from human_framing import (
    add_agreement_columns,
    compare_annotators_to_reference,
    export_agreement_outputs,
    filter_agreed_sentences,
    load_human_framing_annotations,
    load_reference_labels,
    make_label_matrix,
    summarize_agreement,
)

DATA_DIR = REPO_ROOT / 'data' / 'Human expert framing labels' / 'collected_data'
REFERENCE_PATH = REPO_ROOT / 'data' / 'Human expert framing labels' / 'data_colletion_tool' / 'gold_annotation' / '_master_with_reference.csv'
OUTPUT_DIR = REPO_ROOT / 'data' / 'Human expert framing labels' / 'processed_data'

def relpath(path):
    return Path(path).resolve().relative_to(REPO_ROOT.resolve()).as_posix()

pd.set_option('display.max_colwidth', 140)
print(f'Input data: {relpath(DATA_DIR)}')
print(f'AI reference labels: {relpath(REFERENCE_PATH)}')
print(f'Output data: {relpath(OUTPUT_DIR)}')


Input data: data/Human expert framing labels/collected_data
AI reference labels: data/Human expert framing labels/data_colletion_tool/gold_annotation/_master_with_reference.csv
Output data: data/Human expert framing labels/processed_data


## Load Collected Labels

In [28]:
annotations = load_human_framing_annotations(DATA_DIR)

print(f'Rows: {len(annotations):,}')
print(f'Annotators: {annotations["annotator"].nunique()}')
print(f'Sentences: {annotations["item_id"].nunique()}')

annotations.head()

Rows: 120
Annotators: 3
Sentences: 40


,annotator,annotator_name,item_id,country,sentence,gold_label,source_file
0,1,NaN,S01,Canada,"Its generative Al tool, AgPal Chat, helps users find relevant funding and resources faster, supporting the sustainable growth and compet...",Neutral,annotator_1.csv
1,1,NaN,S02,China,China’s R&D expenditure is approaching that of the whole EU.” China’s Strengths Creates Innovation Miracles Good policy is like sunshine.,Innovation-oriented,annotator_1.csv
2,1,NaN,S03,France,"for the adoption of AI by businesses), contractual relationships with private partners (e.g.",Unsure,annotator_1.csv
3,1,NaN,S04,Germany,"The Federal Government will therefore further expand its funding of applied research, development and testing of complex autonomous driv...",Innovation-oriented,annotator_1.csv
4,1,NaN,S05,Italy,"In this sense, at least two sets of problems arise, upon which the space and competitiveness of Artificial Intelligence systems develope...",Neutral,annotator_1.csv


In [29]:
label_distribution = (
    annotations
    .groupby('annotator')['gold_label']
    .value_counts(dropna=False)
    .rename('count')
    .reset_index()
    .sort_values(['annotator', 'gold_label'])
)

label_distribution

,annotator,gold_label,count
1,1,Innovation-oriented,11
3,1,Mixed,4
0,1,Neutral,11
2,1,Risk-oriented,8
4,1,Unsure,3
5,1,NaN,3
8,2,Innovation-oriented,8
6,2,Mixed,9
10,2,Neutral,6
9,2,Risk-oriented,8


## AI Reference Labels

The master file contains model reference labels. These are mapped to the human label scheme before comparison: `AI-friendly` becomes `Innovation-oriented`, `AI-cautious` becomes `Risk-oriented`, `mixed` becomes `Mixed`, and `neutral` becomes `Neutral`.

In [30]:
reference_labels = load_reference_labels(REFERENCE_PATH)

reference_labels['reference_label'].value_counts().rename('count').reset_index()


,reference_label,count
0,Innovation-oriented,10
1,Risk-oriented,10
2,Mixed,10
3,Neutral,10


## Sentence-Level Agreement

In [31]:
label_matrix = make_label_matrix(annotations)
agreement_matrix = add_agreement_columns(label_matrix)

agreement_summary = summarize_agreement(agreement_matrix)
agreement_summary

,agreement_share,majority_label,sentence_count
10,1.000000,Neutral,2
11,1.000000,Risk-oriented,2
8,1.000000,Innovation-oriented,1
9,1.000000,Mixed,1
12,1.000000,Unsure,1
3,0.666667,Innovation-oriented,10
6,0.666667,Risk-oriented,7
4,0.666667,Mixed,5
7,0.666667,Unsure,5
5,0.666667,Neutral,2


## Annotators vs AI Reference

In [32]:
annotator_reference_comparison = compare_annotators_to_reference(label_matrix, reference_labels)
annotator_reference_comparison


,annotator,compared_sentences,matches_reference,reference_match_rate
2,4,40,24,0.600000
1,2,40,20,0.500000
0,1,37,14,0.378378


In [33]:
reference_comparison_matrix = agreement_matrix.merge(
    reference_labels[['item_id', 'country', 'sentence', 'model_label_reference', 'reference_label']],
    on=['item_id', 'country', 'sentence'],
    how='left',
)

reference_comparison_matrix['majority_matches_reference'] = (
    reference_comparison_matrix['majority_label'] == reference_comparison_matrix['reference_label']
)

reference_comparison_matrix[[
    'item_id', 'country', 'sentence', 'model_label_reference', 'reference_label',
    'majority_label', 'majority_count', 'agreement_share', 'majority_matches_reference'
]].head()


,item_id,country,sentence,model_label_reference,reference_label,majority_label,majority_count,agreement_share,majority_matches_reference
0,S01,Canada,"Its generative Al tool, AgPal Chat, helps users find relevant funding and resources faster, supporting the sustainable growth and compet...",AI-friendly,Innovation-oriented,Innovation-oriented,2,0.666667,True
1,S02,China,China’s R&D expenditure is approaching that of the whole EU.” China’s Strengths Creates Innovation Miracles Good policy is like sunshine.,AI-friendly,Innovation-oriented,Innovation-oriented,2,0.666667,True
2,S03,France,"for the adoption of AI by businesses), contractual relationships with private partners (e.g.",AI-friendly,Innovation-oriented,Unsure,3,1.000000,False
3,S04,Germany,"The Federal Government will therefore further expand its funding of applied research, development and testing of complex autonomous driv...",AI-friendly,Innovation-oriented,Innovation-oriented,2,0.666667,True
4,S05,Italy,"In this sense, at least two sets of problems arise, upon which the space and competitiveness of Artificial Intelligence systems develope...",AI-friendly,Innovation-oriented,Unsure,2,0.666667,False


In [34]:
full_agreement = filter_agreed_sentences(annotations, min_agreement_share=1.0)
full_agreement = full_agreement.merge(
    reference_labels[['item_id', 'country', 'sentence', 'model_label_reference', 'reference_label']],
    on=['item_id', 'country', 'sentence'],
    how='left',
)
full_agreement['majority_matches_reference'] = full_agreement['majority_label'] == full_agreement['reference_label']

print(f'Fully agreed sentences: {len(full_agreement)} / {agreement_matrix.shape[0]}')
full_agreement[['item_id', 'country', 'sentence', 'model_label_reference', 'majority_label', 'agreement_share', 'majority_matches_reference']]

Fully agreed sentences: 7 / 40


,item_id,country,sentence,model_label_reference,majority_label,agreement_share,majority_matches_reference
0,S03,France,"for the adoption of AI by businesses), contractual relationships with private partners (e.g.",AI-friendly,Unsure,1.0,False
1,S10,China,"Innovation is “fl owing water”, making China’s seed of advanced ideas into towering trees.",AI-friendly,Innovation-oriented,1.0,True
2,S11,Canada,Respondents suggested regular ethical audits to ensure compliance with the guidelines and prevent harmful biases.,AI-cautious,Risk-oriented,1.0,True
3,S15,Italy,"In particular, the Committee will take care to highlight the risks inherent to specific initiatives, guiding project choices towards app...",AI-cautious,Risk-oriented,1.0,True
4,S19,Canada,"Some law or policy is silent on areas needed to govern AI, while others may introduce unintended bias in data collection or obstruct AI ...",AI-cautious,Mixed,1.0,False
5,S33,France, The public reports of the Court of Auditors are available online on the website of the Court and the regional and territorial chambers...,neutral,Neutral,1.0,True
6,S38,United States,"These agencies could then provide analysis of AI adoption, job creation, displacement, and wage effects.",neutral,Neutral,1.0,True


Because annotator 1 is less stable, the main gold set below excludes annotator 1 and keeps sentences where annotator 2 Daria and annotator 4 Stephen agree. The earlier 3/3 and 2/3 sets remain useful as sensitivity checks.

In [35]:
# Optional majority-agreement set for sensitivity checks.
majority_agreement = filter_agreed_sentences(annotations, min_agreement_share=2/3)
majority_agreement = majority_agreement.merge(
    reference_labels[['item_id', 'country', 'sentence', 'model_label_reference', 'reference_label']],
    on=['item_id', 'country', 'sentence'],
    how='left',
)
majority_agreement['majority_matches_reference'] = majority_agreement['majority_label'] == majority_agreement['reference_label']

print(f'Majority-agreed sentences: {len(majority_agreement)} / {agreement_matrix.shape[0]}')
majority_agreement[['item_id', 'country', 'sentence', 'model_label_reference', 'majority_label', 'agreement_share', 'majority_matches_reference']].head()

Majority-agreed sentences: 36 / 40


,item_id,country,sentence,model_label_reference,majority_label,agreement_share,majority_matches_reference
0,S01,Canada,"Its generative Al tool, AgPal Chat, helps users find relevant funding and resources faster, supporting the sustainable growth and compet...",AI-friendly,Innovation-oriented,0.666667,True
1,S02,China,China’s R&D expenditure is approaching that of the whole EU.” China’s Strengths Creates Innovation Miracles Good policy is like sunshine.,AI-friendly,Innovation-oriented,0.666667,True
2,S03,France,"for the adoption of AI by businesses), contractual relationships with private partners (e.g.",AI-friendly,Unsure,1.000000,False
3,S04,Germany,"The Federal Government will therefore further expand its funding of applied research, development and testing of complex autonomous driv...",AI-friendly,Innovation-oriented,0.666667,True
4,S05,Italy,"In this sense, at least two sets of problems arise, upon which the space and competitiveness of Artificial Intelligence systems develope...",AI-friendly,Unsure,0.666667,False


## Main Gold Set: Daria and Stephen Agreement

In [36]:
trusted_annotations = annotations[annotations['annotator'].isin([2, 4])].copy()

daria_stephen_agreement = filter_agreed_sentences(trusted_annotations, min_agreement_share=1.0)
daria_stephen_agreement = daria_stephen_agreement.merge(
    reference_labels[['item_id', 'country', 'sentence', 'model_label_reference', 'reference_label']],
    on=['item_id', 'country', 'sentence'],
    how='left',
)
daria_stephen_agreement['majority_matches_reference'] = (
    daria_stephen_agreement['majority_label'] == daria_stephen_agreement['reference_label']
)

print(f'Daria + Stephen agreed sentences: {len(daria_stephen_agreement)} / {agreement_matrix.shape[0]}')
reference_match_count = daria_stephen_agreement['majority_matches_reference'].sum()
print(f'Match AI reference: {reference_match_count} / {len(daria_stephen_agreement)}')
daria_stephen_agreement[[
    'item_id', 'country', 'sentence', 'model_label_reference',
    'majority_label', 'agreement_share', 'majority_matches_reference'
]]


Daria + Stephen agreed sentences: 23 / 40
Match AI reference: 15 / 23


,item_id,country,sentence,model_label_reference,majority_label,agreement_share,majority_matches_reference
0,S01,Canada,"Its generative Al tool, AgPal Chat, helps users find relevant funding and resources faster, supporting the sustainable growth and compet...",AI-friendly,Innovation-oriented,1.0,True
1,S03,France,"for the adoption of AI by businesses), contractual relationships with private partners (e.g.",AI-friendly,Unsure,1.0,False
2,S05,Italy,"In this sense, at least two sets of problems arise, upon which the space and competitiveness of Artificial Intelligence systems develope...",AI-friendly,Unsure,1.0,False
3,S06,Japan,"In the field of Al, where basic research and social implementation are closely connected, the lack of progress in implementation has bec...",AI-friendly,Innovation-oriented,1.0,True
4,S07,United Kingdom,oxinote 3) Setting a short-term target to train tens of thousands of Al professionals by 2030 will help bridge the estimated gap between...,AI-friendly,Innovation-oriented,1.0,True
5,S10,China,"Innovation is “fl owing water”, making China’s seed of advanced ideas into towering trees.",AI-friendly,Innovation-oriented,1.0,True
6,S11,Canada,Respondents suggested regular ethical audits to ensure compliance with the guidelines and prevent harmful biases.,AI-cautious,Risk-oriented,1.0,True
7,S13,France,Tailor-made treatment should be reserved for public procurement involving the processing of sensitive data.,AI-cautious,Risk-oriented,1.0,True
8,S14,Germany,It is also working to ensure that all stakeholders in the field of AI honour their individual responsibility to respect human rights.,AI-cautious,Risk-oriented,1.0,True
9,S15,Italy,"In particular, the Committee will take care to highlight the risks inherent to specific initiatives, guiding project choices towards app...",AI-cautious,Risk-oriented,1.0,True


## Export Processed Data

In [37]:
main_exported = export_agreement_outputs(
    daria_stephen_agreement,
    OUTPUT_DIR,
    stem='human_expert_framing_daria_stephen_agreement',
)

{name: relpath(path) for name, path in main_exported.items()}


{'agreed_sentences': 'data/Human expert framing labels/processed_data/human_expert_framing_daria_stephen_agreement_agreed_sentences.csv',
 'agreement_summary': 'data/Human expert framing labels/processed_data/human_expert_framing_daria_stephen_agreement_agreement_summary.csv'}

In [38]:
exported = export_agreement_outputs(
    full_agreement,
    OUTPUT_DIR,
    stem='human_expert_framing_full_agreement',
)

{name: relpath(path) for name, path in exported.items()}


{'agreed_sentences': 'data/Human expert framing labels/processed_data/human_expert_framing_full_agreement_agreed_sentences.csv',
 'agreement_summary': 'data/Human expert framing labels/processed_data/human_expert_framing_full_agreement_agreement_summary.csv'}

In [39]:
# Export the majority-agreement set separately so it is available but not confused with the strict gold set.
majority_exported = export_agreement_outputs(
    majority_agreement,
    OUTPUT_DIR,
    stem='human_expert_framing_majority_agreement',
)

{name: relpath(path) for name, path in majority_exported.items()}


{'agreed_sentences': 'data/Human expert framing labels/processed_data/human_expert_framing_majority_agreement_agreed_sentences.csv',
 'agreement_summary': 'data/Human expert framing labels/processed_data/human_expert_framing_majority_agreement_agreement_summary.csv'}